# Laser Control

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")
print(f"Python version: {sys.version}")

In [ ]:
from femtum_sdk.api.contracts.v1.hardware.health_pb2 import HealthResponse, HealthRequest
from femtum_sdk.api import HardwareApi

api_host = "localhost:5210"

with HardwareApi(host=api_host) as api:
    result: HealthResponse = api.health.GetHealth(HealthRequest())
    print(result)


## Fetch Laser Parameters

In [ ]:
from femtum_sdk.api.contracts.v1.dto.femtosecond_laser_pb2 import GatingMode, LaserStatus
from femtum_sdk.api.contracts.v1.hardware.femtosecond_laser_pb2 import (
    GetLaserParametersRequest,
    GetLaserParametersResponse,
)

with HardwareApi(host=api_host) as api:
    response: GetLaserParametersResponse = api.femtosecond_laser.GetParameters(GetLaserParametersRequest())
    parameters = response.Parameters

    print(f"IsEnabled: {parameters.IsEnabled}")
    print(f"PowerLevelPercent: {parameters.PowerLevelPercent}")
    print(f"GateControl: {GatingMode.Name(parameters.GateControl)} ({parameters.GateControl})")
    print(f"PulseRepetitionRateKhz: {parameters.PulseRepetitionRateKhz}")
    print(f"RepetitionRateDivider: {parameters.RepetitionRateDivider}")
    print(f"PulseDurationFs: {parameters.PulseDurationFs}")
    print(f"PulseDispersionFs2: {parameters.PulseDispersionFs2}")
    print(f"BurstLength: {parameters.BurstLength}")
    print(f"Status: {LaserStatus.Name(parameters.Status)} ({parameters.Status})")
    print(f"LaserHeadTemperatureCelsius: {parameters.LaserHeadTemperatureCelsius}")
    print(f"LaserInOperationTotalTimeSeconds: {parameters.LaserInOperationTotalTimeSeconds}")
    print(f"Error: {parameters.Error!r}")
    print(f"Firmware: {parameters.Firmware!r}")
    print(f"SerialNumber: {parameters.SerialNumber!r}")



## Assign Parameters

In [ ]:
from femtum_sdk.api.contracts.v1.dto.femtosecond_laser_pb2 import FemtosecondLaserInputParameters
from femtum_sdk.api.contracts.v1.hardware.femtosecond_laser_pb2 import ApplyLaserParametersRequest, ApplyLaserParametersResponse

with HardwareApi(host=api_host) as api:
    # Fetch initial parameters
    response: GetLaserParametersResponse = api.femtosecond_laser.GetParameters(GetLaserParametersRequest())
    initial_parameters = response.Parameters

    # Apply new parameters
    response: ApplyLaserParametersResponse = api.femtosecond_laser.ApplyParameters(ApplyLaserParametersRequest(Parameters=FemtosecondLaserInputParameters(
        PowerLevelPercent=0,
        GateControl=GatingMode.GATING_MODE_GATE_ACTIVE_HIGH,
    )))
    parameters = response.Parameters

    # Fetch applied parameters
    response: GetLaserParametersResponse = api.femtosecond_laser.GetParameters(GetLaserParametersRequest())
    parameters = response.Parameters

    assert parameters.PowerLevelPercent == 0
    assert parameters.GateControl == GatingMode.GATING_MODE_GATE_ACTIVE_HIGH

## Cleanup

The connection is closed automatically when each `with` block exits, so no manual cleanup is required.